# SINet: Daily Predictions of F10.7 and F30 Solar Indices — Implementation / 구현

**논문 / Paper**: Wang et al., "Daily Predictions of F10.7 and F30 Solar Indices With Deep Learning," *Space Weather*, 2026.

이 노트북은 Wang et al. (2026)이 제안한 **SINet (Solar Index Network)** 의 핵심 구조를 PyTorch로 구현합니다.
FFT 기반 주기 탐지, Dual-Inception Block, 적응적 집계(Adaptive Aggregation) 등 논문의 주요 아이디어를 교육 목적으로 재현합니다.
합성 F10.7 데이터를 사용하여 외부 데이터 파일 없이 독립 실행이 가능합니다.

This notebook implements the core architecture of **SINet (Solar Index Network)** proposed by Wang et al. (2026) using PyTorch.
We reproduce the paper's key ideas — FFT-based period detection, Dual-Inception Blocks, and Adaptive Aggregation — for educational purposes.
Synthetic F10.7 data is used so the notebook runs standalone without external data files.

**구현 파트 / Implementation Parts**:
1. 데이터 생성 및 전처리 / Data Generation & Preprocessing
2. 고정 vs 롤링 예측 데이터 레이블링 / Fixed vs Rolling Prediction Labeling
3. SINet 아키텍처 / SINet Architecture
4. 학습 및 평가 / Training & Evaluation
5. 시각화 / Visualization
6. 자기상관 분석 / Autocorrelation Analysis

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: 합성 F10.7 데이터 생성 및 전처리 / Synthetic F10.7 Data Generation & Preprocessing

실제 F10.7 데이터는 NOAA SWPC(https://www.swpc.noaa.gov/phenomena/f107-cm-radio-emissions)에서 공개적으로 제공됩니다.
이 노트북에서는 교육 목적으로 실제 F10.7의 특성을 모방한 합성 데이터를 생성합니다:

Real F10.7 data is publicly available from NOAA SWPC. For educational purposes, we generate synthetic data that mimics key F10.7 characteristics:

- **11년 태양 주기 / 11-year solar cycle**: 장기 주기적 변동 / Long-term periodic variation (~4018 days)
- **~27일 태양 자전 / ~27-day solar rotation**: 단기 변조 / Short-term modulation
- **범위 / Range**: 50–400 sfu (solar flux units)
- **잡음 / Noise**: 실제 관측 불확실성 모사 / Simulates observational uncertainty

**Min-Max 정규화 / Min-Max Normalization**:

$$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}}$$

In [ ]:
def generate_synthetic_f107(n_days: int = 8000) -> np.ndarray:
    """Generate synthetic F10.7 data mimicking real solar flux characteristics.

    The synthetic signal combines an 11-year solar cycle envelope with
    a ~27-day rotational modulation plus Gaussian noise, clipped to the
    realistic F10.7 range of 50-400 sfu.

    Args:
        n_days: Number of daily data points to generate.

    Returns:
        Array of synthetic F10.7 values in sfu.
    """
    t = np.arange(n_days, dtype=np.float64)

    # 11-year solar cycle (~4018 days)
    solar_cycle = 120 + 100 * np.sin(2 * np.pi * t / 4018)

    # ~27-day solar rotation modulation (amplitude scales with cycle phase)
    rotation_amp = 10 + 15 * (1 + np.sin(2 * np.pi * t / 4018)) / 2
    rotation = rotation_amp * np.sin(2 * np.pi * t / 27.0)

    # Additional harmonic (~13.5 day half-rotation)
    half_rotation = 5 * np.sin(2 * np.pi * t / 13.5)

    # Gaussian noise
    noise = np.random.normal(0, 8, n_days)

    f107 = solar_cycle + rotation + half_rotation + noise
    f107 = np.clip(f107, 50, 400)

    return f107


# Generate data
f107_data = generate_synthetic_f107(n_days=8000)

print(f"Data length: {len(f107_data)} days (~{len(f107_data)/365.25:.1f} years)")
print(f"Range: [{f107_data.min():.1f}, {f107_data.max():.1f}] sfu")
print(f"Mean: {f107_data.mean():.1f} sfu, Std: {f107_data.std():.1f} sfu")

In [ ]:
# Min-Max normalization
f107_min, f107_max = f107_data.min(), f107_data.max()
f107_norm = (f107_data - f107_min) / (f107_max - f107_min)

# Train / Test split (80/20, chronological)
split_idx = int(len(f107_norm) * 0.8)
train_data = f107_norm[:split_idx]
test_data = f107_norm[split_idx:]

print(f"Train: {len(train_data)} days, Test: {len(test_data)} days")
print(f"Normalized range: [{f107_norm.min():.4f}, {f107_norm.max():.4f}]")

## Part 2: 고정 vs 롤링 예측 데이터 레이블링 / Fixed vs Rolling Prediction Data Labeling

논문은 두 가지 예측 모드를 비교합니다:

The paper compares two prediction modes:

| 모드 / Mode | 입력 / Input | 출력 / Output | 설명 / Description |
|---|---|---|---|
| **고정 예측 / Fixed** | $[d_{-29}, \ldots, d_0]$ (30일) | $[d_1, \ldots, d_{60}]$ (60일) | 한 번에 60일 전체를 예측 / Predict all 60 days at once |
| **롤링 예측 / Rolling** | $[d_{-29}, \ldots, d_0]$ (30일) | $d_1$ (1일) | 예측값을 입력에 반복 추가하며 반복 / Iteratively append predicted value and repeat |

고정 예측은 모델이 직접 다중 스텝을 학습하므로 오류 누적이 없지만, 장기 예측에서 정확도가 떨어질 수 있습니다.
롤링 예측은 단기에 정확하지만, 예측 오류가 반복적으로 누적됩니다.

Fixed prediction trains the model to output all steps directly (no error accumulation), but may lose accuracy at longer horizons.
Rolling prediction is accurate short-term but accumulates prediction errors iteratively.

In [ ]:
class SolarIndexDataset(Dataset):
    """PyTorch Dataset for solar index time series prediction.

    Creates (input, label) pairs from a 1D time series using a sliding window.
    Supports both fixed (multi-step) and rolling (single-step) prediction modes.

    Args:
        data: Normalized 1D time series array.
        input_len: Number of past days used as input (default 30).
        output_len: Number of future days to predict (default 60).
        mode: 'fixed' for multi-step output, 'rolling' for single-step.
    """

    def __init__(
        self,
        data: np.ndarray,
        input_len: int = 30,
        output_len: int = 60,
        mode: str = "fixed",
    ):
        self.data = data.astype(np.float32)
        self.input_len = input_len
        self.output_len = output_len
        self.mode = mode

    def __len__(self) -> int:
        if self.mode == "fixed":
            return len(self.data) - self.input_len - self.output_len + 1
        else:  # rolling: only need 1 step ahead
            return len(self.data) - self.input_len

    def __getitem__(self, idx: int):
        x = self.data[idx : idx + self.input_len]
        if self.mode == "fixed":
            y = self.data[idx + self.input_len : idx + self.input_len + self.output_len]
        else:
            y = self.data[idx + self.input_len : idx + self.input_len + 1]
        return torch.tensor(x).unsqueeze(0), torch.tensor(y)  # (1, input_len), (output_len,)


# Create datasets
INPUT_LEN = 30
OUTPUT_LEN = 60
BATCH_SIZE = 32

train_dataset = SolarIndexDataset(train_data, INPUT_LEN, OUTPUT_LEN, mode="fixed")
test_dataset = SolarIndexDataset(test_data, INPUT_LEN, OUTPUT_LEN, mode="fixed")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

x_sample, y_sample = train_dataset[0]
print(f"Input shape: {x_sample.shape}  (channels=1, seq_len={INPUT_LEN})")
print(f"Output shape: {y_sample.shape}  (forecast horizon={OUTPUT_LEN})")
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

## Part 3: SINet 아키텍처 / SINet Architecture

SINet의 핵심 구성 요소는 **TimesBlock**이며, 다음과 같은 단계로 작동합니다:

The core building block of SINet is the **TimesBlock**, which operates as follows:

### 3-1. FFT 주기 탐지 / FFT Period Detection
입력 시계열에 FFT를 적용하여 상위 $k=3$ 주기를 추출하고, 해당 진폭을 softmax로 가중치화합니다.

Apply FFT to the input time series, extract top-$k=3$ periods by amplitude, and compute softmax weights from those amplitudes.

### 3-2. 2D 리셰이핑 / 2D Reshaping
각 탐지된 주기 $p_i$에 대해, 1D 시계열 $(B, C, T)$을 2D 텐서 $(B, C, p_i, \lceil T/p_i \rceil)$로 리셰이핑합니다.

For each detected period $p_i$, reshape the 1D series $(B, C, T)$ into a 2D tensor $(B, C, p_i, \lceil T/p_i \rceil)$.

### 3-3. Dual-Inception Block
6개의 병렬 Conv2D 경로 (커널 크기: $1{\times}1$, $3{\times}3$, $5{\times}5$, $7{\times}7$, $9{\times}9$, $11{\times}11$)로 구성됩니다.
두 개의 Inception 레이어를 GELU 활성화 함수와 함께 연결합니다.

Six parallel Conv2D paths (kernel sizes: $1{\times}1$, $3{\times}3$, $5{\times}5$, $7{\times}7$, $9{\times}9$, $11{\times}11$).
Two Inception layers connected with GELU activation.

### 3-4. 적응적 집계 및 잔차 연결 / Adaptive Aggregation & Residual
$k$개 주기 출력을 softmax 가중치로 합산한 뒤, 원래 입력을 잔차로 더합니다.

Sum the $k$ period outputs using softmax weights, then add the original input as a residual connection.

In [ ]:
def fft_period_detection(x: torch.Tensor, top_k: int = 3):
    """Detect dominant periods in a time series using FFT.

    Applies real FFT along the time dimension, selects the top-k frequency
    components by amplitude, converts them to periods, and returns softmax
    weights derived from the amplitudes.

    Args:
        x: Input tensor of shape (B, C, T).
        top_k: Number of dominant periods to extract.

    Returns:
        periods: List of int periods (length top_k).
        weights: Softmax weights tensor of shape (top_k,).
    """
    B, C, T = x.shape
    # Average over batch and channel for stable period detection
    x_mean = x.mean(dim=(0, 1))  # (T,)

    # Real FFT — ignore DC component (index 0)
    fft_vals = torch.fft.rfft(x_mean)
    amplitudes = torch.abs(fft_vals)[1:]  # exclude DC

    # Top-k amplitudes → periods
    top_amp, top_idx = torch.topk(amplitudes, top_k)
    top_idx = top_idx + 1  # offset for removed DC component

    # Convert frequency index to period: period = T / freq_index
    periods = [max(2, int(T / idx.item())) for idx in top_idx]

    # Softmax weights from amplitudes
    weights = torch.softmax(top_amp, dim=0)

    return periods, weights


# Demonstrate FFT period detection on a sample batch
x_demo = torch.tensor(train_data[:INPUT_LEN], dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1, 1, 30)
periods, weights = fft_period_detection(x_demo, top_k=3)
print("Detected periods:", periods)
print("Softmax weights:", [f"{w:.3f}" for w in weights.tolist()])

In [ ]:
class InceptionBlock(nn.Module):
    """Single Inception block with 6 parallel Conv2D paths.

    Each path uses a different kernel size (1x1 through 11x11) to capture
    multi-scale spatial patterns. Outputs are summed element-wise.

    Args:
        in_channels: Number of input channels.
        out_channels: Number of output channels per path.
    """

    KERNEL_SIZES = [1, 3, 5, 7, 9, 11]

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv2d(
                in_channels, out_channels,
                kernel_size=k, padding=k // 2
            )
            for k in self.KERNEL_SIZES
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass — sum of all parallel conv paths.

        Args:
            x: Input tensor of shape (B, C_in, H, W).

        Returns:
            Output tensor of shape (B, C_out, H, W).
        """
        return sum(conv(x) for conv in self.convs)


class DualInceptionBlock(nn.Module):
    """Two cascaded Inception blocks with GELU activation in between.

    Architecture: InceptionBlock(C_in→C_mid) → GELU → InceptionBlock(C_mid→C_out)

    Args:
        in_channels: Input channels.
        mid_channels: Intermediate channels (default 64).
        out_channels: Output channels.
    """

    def __init__(self, in_channels: int, mid_channels: int = 64, out_channels: int = 32):
        super().__init__()
        self.inception1 = InceptionBlock(in_channels, mid_channels)
        self.gelu = nn.GELU()
        self.inception2 = InceptionBlock(mid_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.inception1(x)
        x = self.gelu(x)
        x = self.inception2(x)
        return x


class TimesBlock(nn.Module):
    """Core building block of SINet.

    Performs FFT-based period detection, reshapes 1D→2D for each period,
    applies DualInceptionBlock, then adaptively aggregates results with
    a residual connection.

    Args:
        d_model: Model dimension (number of channels).
        top_k: Number of dominant periods to detect via FFT.
        dropout: Dropout rate.
    """

    def __init__(self, d_model: int = 32, top_k: int = 3, dropout: float = 0.3):
        super().__init__()
        self.top_k = top_k
        self.d_model = d_model
        self.dual_inception = DualInceptionBlock(d_model, 64, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with FFT period detection and adaptive aggregation.

        Args:
            x: Input tensor of shape (B, C, T) where C = d_model.

        Returns:
            Output tensor of shape (B, C, T).
        """
        B, C, T = x.shape
        residual = x

        # FFT period detection
        periods, weights = fft_period_detection(x, self.top_k)

        aggregated = torch.zeros_like(x)

        for i, period in enumerate(periods):
            # Pad time dimension to be divisible by period
            n_cols = (T + period - 1) // period
            pad_len = n_cols * period - T
            x_padded = torch.nn.functional.pad(x, (0, pad_len))

            # Reshape 1D → 2D: (B, C, T_pad) → (B, C, period, n_cols)
            x_2d = x_padded.reshape(B, C, period, n_cols)

            # Apply Dual Inception Block
            out_2d = self.dual_inception(x_2d)

            # Reshape back to 1D and trim padding
            out_1d = out_2d.reshape(B, C, -1)[:, :, :T]

            aggregated += weights[i] * out_1d

        # Residual connection + dropout
        output = self.dropout(aggregated) + residual
        return output


# Quick shape test
block = TimesBlock(d_model=32, top_k=3)
test_input = torch.randn(4, 32, 30)
test_output = block(test_input)
print(f"TimesBlock: {test_input.shape} → {test_output.shape}")

In [ ]:
class SINet(nn.Module):
    """Solar Index Network for F10.7 / F30 prediction.

    Architecture:
        Linear embedding (1 → d_model) → 2 x TimesBlock → LayerNorm → Linear projection (d_model → 1)
        → Flatten → Linear (input_len → output_len)

    Configuration follows Table 2 of Wang et al. (2026):
        d_model=32, top_k=3, n_blocks=2, dropout=0.3

    Args:
        input_len: Number of input time steps (default 30).
        output_len: Number of output time steps (default 60).
        d_model: Hidden dimension (default 32).
        n_blocks: Number of stacked TimesBlocks (default 2).
        top_k: Number of FFT periods (default 3).
        dropout: Dropout rate (default 0.3).
    """

    def __init__(
        self,
        input_len: int = 30,
        output_len: int = 60,
        d_model: int = 32,
        n_blocks: int = 2,
        top_k: int = 3,
        dropout: float = 0.3,
    ):
        super().__init__()
        self.input_len = input_len
        self.output_len = output_len
        self.d_model = d_model

        # Embedding: expand channel 1 → d_model
        self.embedding = nn.Linear(1, d_model)

        # Stacked TimesBlocks
        self.times_blocks = nn.ModuleList([
            TimesBlock(d_model=d_model, top_k=top_k, dropout=dropout)
            for _ in range(n_blocks)
        ])

        self.layer_norm = nn.LayerNorm(d_model)

        # Projection: d_model → 1, then flatten → output mapping
        self.channel_proj = nn.Linear(d_model, 1)
        self.output_proj = nn.Linear(input_len, output_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass.

        Args:
            x: Input tensor of shape (B, 1, input_len).

        Returns:
            Predictions of shape (B, output_len).
        """
        B = x.shape[0]

        # (B, 1, T) → (B, T, 1) → embedding → (B, T, d_model) → (B, d_model, T)
        x = x.squeeze(1).unsqueeze(-1)        # (B, T, 1)
        x = self.embedding(x)                  # (B, T, d_model)
        x = x.permute(0, 2, 1)                 # (B, d_model, T)

        # Stacked TimesBlocks
        for block in self.times_blocks:
            x = block(x)

        # (B, d_model, T) → (B, T, d_model) → LayerNorm → project → (B, T, 1)
        x = x.permute(0, 2, 1)
        x = self.layer_norm(x)
        x = self.channel_proj(x).squeeze(-1)   # (B, T)

        # Map input_len → output_len
        out = self.output_proj(x)               # (B, output_len)
        return out


# Instantiate and summarize
model = SINet(
    input_len=INPUT_LEN,
    output_len=OUTPUT_LEN,
    d_model=32,
    n_blocks=2,
    top_k=3,
    dropout=0.3,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel architecture:\n{model}")

## Part 4: 학습 및 평가 / Training & Evaluation

논문의 Table 2에 따른 학습 설정:

Training configuration from Table 2 of the paper:

| 파라미터 / Parameter | 값 / Value |
|---|---|
| Loss function | MSE (Mean Squared Error) |
| Optimizer | Adam |
| Learning rate | 0.001 |
| Batch size | 32 |
| Epochs | 10 |
| Dropout | 0.3 |

**평가 지표 / Evaluation Metrics**:

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i - y_i)^2}$$

$$\text{MAE} = \frac{1}{N}\sum_{i=1}^{N}|\hat{y}_i - y_i|$$

$$\text{MAPE} = \frac{1}{N}\sum_{i=1}^{N}\frac{|\hat{y}_i - y_i|}{y_i} \times 100\%$$

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Compute RMSE, MAE, and MAPE between true and predicted values.

    All inputs should be in original (denormalized) scale for meaningful
    MAPE computation.

    Args:
        y_true: Ground truth array.
        y_pred: Predicted array.

    Returns:
        Dictionary with 'rmse', 'mae', 'mape' keys.
    """
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))
    mae = np.mean(np.abs(y_pred - y_true))
    # Avoid division by zero in MAPE
    mask = y_true > 1e-8
    mape = np.mean(np.abs((y_pred[mask] - y_true[mask]) / y_true[mask])) * 100
    return {"rmse": rmse, "mae": mae, "mape": mape}


def denormalize(x_norm: np.ndarray, x_min: float, x_max: float) -> np.ndarray:
    """Reverse min-max normalization.

    Args:
        x_norm: Normalized values in [0, 1].
        x_min: Original minimum value.
        x_max: Original maximum value.

    Returns:
        Values in original scale.
    """
    return x_norm * (x_max - x_min) + x_min

In [ ]:
# Training loop
EPOCHS = 10
LR = 0.001

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        pred = model(x_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_batches += 1

    avg_train_loss = epoch_loss / n_batches
    train_losses.append(avg_train_loss)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    n_val = 0
    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            pred = model(x_batch)
            val_loss += criterion(pred, y_batch).item()
            n_val += 1

    avg_val_loss = val_loss / n_val
    val_losses.append(avg_val_loss)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | "
          f"Train Loss: {avg_train_loss:.6f} | "
          f"Val Loss: {avg_val_loss:.6f}")

In [ ]:
# Collect predictions on the test set
model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        pred = model(x_batch)
        all_preds.append(pred.cpu().numpy())
        all_targets.append(y_batch.numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_targets = np.concatenate(all_targets, axis=0)

# Denormalize to original sfu scale
preds_sfu = denormalize(all_preds, f107_min, f107_max)
targets_sfu = denormalize(all_targets, f107_min, f107_max)

# Compute metrics per forecast horizon
horizons = [1, 27, 45, 60]
print("Evaluation metrics per forecast horizon (sfu):")
print("-" * 55)
print(f"{'Horizon':>10s} {'RMSE':>10s} {'MAE':>10s} {'MAPE (%)':>10s}")
print("-" * 55)

horizon_metrics = {}
for h in horizons:
    m = compute_metrics(targets_sfu[:, h - 1], preds_sfu[:, h - 1])
    horizon_metrics[h] = m
    print(f"{h:>10d} {m['rmse']:>10.2f} {m['mae']:>10.2f} {m['mape']:>10.2f}")

# Overall metrics (all 60 steps)
overall = compute_metrics(targets_sfu.flatten(), preds_sfu.flatten())
print("-" * 55)
print(f"{'Overall':>10s} {overall['rmse']:>10.2f} {overall['mae']:>10.2f} {overall['mape']:>10.2f}")

## Part 5: 시각화 / Visualization

논문의 주요 그림을 재현합니다:

We reproduce the paper's key figures:

1. **합성 F10.7 시계열 + 학습/테스트 분할** / Synthetic F10.7 time series with train/test split (cf. Figure 1)
2. **학습 손실 곡선** / Training loss curves
3. **예측 vs 관측 (다양한 예측 수평선)** / Predicted vs observed at different forecast horizons (cf. Figure 6)
4. **예측 수평선별 오류** / Prediction error over forecast horizon (cf. Figure 10)
5. **수평선별 지표 막대 그래프** / Bar chart of metrics at key horizons

In [ ]:
# --- Figure 1: Synthetic F10.7 time series with train/test split ---
fig, ax = plt.subplots(figsize=(14, 4))
days = np.arange(len(f107_data))
ax.plot(days[:split_idx], f107_data[:split_idx], color='steelblue', linewidth=0.5, label='Train')
ax.plot(days[split_idx:], f107_data[split_idx:], color='coral', linewidth=0.5, label='Test')
ax.axvline(split_idx, color='black', linestyle='--', linewidth=1, label='Train/Test split')
ax.set_xlabel('Day')
ax.set_ylabel('F10.7 (sfu)')
ax.set_title('Synthetic F10.7 Time Series / 합성 F10.7 시계열')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

# --- Figure 2: Training loss curves ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, EPOCHS + 1), train_losses, 'o-', label='Train Loss')
ax.plot(range(1, EPOCHS + 1), val_losses, 's-', label='Validation Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Training & Validation Loss / 학습 및 검증 손실')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 3: Predicted vs Observed at different horizons (cf. Figure 6) ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
sample_range = slice(0, 200)  # Show first 200 test samples

for ax, h in zip(axes.flat, horizons):
    obs = targets_sfu[sample_range, h - 1]
    pred = preds_sfu[sample_range, h - 1]
    ax.plot(obs, color='steelblue', linewidth=0.8, label='Observed / 관측')
    ax.plot(pred, color='coral', linewidth=0.8, label='Predicted / 예측')
    m = compute_metrics(obs, pred)
    ax.set_title(f'{h}-day Forecast / {h}일 예측\n'
                 f'RMSE={m["rmse"]:.1f}, MAE={m["mae"]:.1f}')
    ax.set_xlabel('Sample index')
    ax.set_ylabel('F10.7 (sfu)')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Observed F10.7 at Key Horizons\n'
             '주요 예측 수평선에서의 예측 vs 관측', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 4: RMSE over all 60 forecast horizons (cf. Figure 10) ---
rmse_by_horizon = []
mae_by_horizon = []
for h in range(1, OUTPUT_LEN + 1):
    m = compute_metrics(targets_sfu[:, h - 1], preds_sfu[:, h - 1])
    rmse_by_horizon.append(m['rmse'])
    mae_by_horizon.append(m['mae'])

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(range(1, OUTPUT_LEN + 1), rmse_by_horizon, 'o-', markersize=3,
         color='steelblue', label='RMSE')
ax1.plot(range(1, OUTPUT_LEN + 1), mae_by_horizon, 's-', markersize=3,
         color='coral', label='MAE')
ax1.set_xlabel('Forecast Horizon (days) / 예측 수평선 (일)')
ax1.set_ylabel('Error (sfu)')
ax1.set_title('Prediction Error vs Forecast Horizon\n예측 오류 vs 예측 수평선')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Mark key horizons
for h in horizons:
    ax1.axvline(h, color='gray', linestyle=':', alpha=0.5)
    ax1.annotate(f'd+{h}', (h, rmse_by_horizon[h - 1]),
                 textcoords="offset points", xytext=(5, 10), fontsize=9)
plt.tight_layout()
plt.show()

# --- Figure 5: Bar chart of metrics at key horizons ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metric_names = ['RMSE', 'MAE', 'MAPE (%)']
metric_keys = ['rmse', 'mae', 'mape']
colors = ['steelblue', 'coral', 'seagreen']

for ax, name, key, color in zip(axes, metric_names, metric_keys, colors):
    vals = [horizon_metrics[h][key] for h in horizons]
    bars = ax.bar([f'd+{h}' for h in horizons], vals, color=color, alpha=0.8)
    ax.set_title(name)
    ax.set_ylabel(name)
    ax.grid(True, alpha=0.3, axis='y')
    # Add value labels on bars
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{v:.1f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Metrics at Key Forecast Horizons / 주요 예측 수평선 지표', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Part 6: 자기상관 분석 / Autocorrelation Analysis

논문에서는 F10.7 시계열의 자기상관(autocorrelation)이 예측 성능에 미치는 영향을 분석합니다.
1일 예측은 높은 자기상관 덕분에 정확하지만, 수평선이 길어질수록 자기상관이 감소하여 예측이 어려워집니다.

The paper analyzes how autocorrelation of the F10.7 time series affects prediction performance.
1-day predictions benefit from high autocorrelation, but longer horizons face decreasing autocorrelation, making prediction harder.

**자기상관 공식 / Autocorrelation formula** at lag $k$:

$$\rho_k = \frac{\sum_{t=k+1}^{N}(y_t - \bar{y})(y_{t-k} - \bar{y})}{\sum_{t=1}^{N}(y_t - \bar{y})^2}$$

주요 래그 / Key lags: $k = 1$ (인접일 / adjacent day), $k = 27$ (태양 자전 / solar rotation), $k = 45$, $k = 60$

In [ ]:
def autocorrelation(y: np.ndarray, max_lag: int = 60) -> np.ndarray:
    """Compute autocorrelation function for lags 0 through max_lag.

    Args:
        y: 1D time series array.
        max_lag: Maximum lag to compute.

    Returns:
        Array of autocorrelation values for lags 0..max_lag.
    """
    y_centered = y - y.mean()
    var = np.sum(y_centered ** 2)
    acf = np.zeros(max_lag + 1)
    for k in range(max_lag + 1):
        acf[k] = np.sum(y_centered[k:] * y_centered[:len(y) - k]) / var
    return acf


# Compute autocorrelation on the test data (original scale)
test_original = f107_data[split_idx:]
acf_vals = autocorrelation(test_original, max_lag=60)

# Report at key lags
print("Autocorrelation at key lags / 주요 래그에서의 자기상관:")
print("-" * 40)
for lag in [1, 27, 45, 60]:
    print(f"  lag = {lag:2d}: rho = {acf_vals[lag]:.4f}")

# Plot full autocorrelation function
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(acf_vals)), acf_vals, color='steelblue', alpha=0.7)
for lag in [1, 27, 45, 60]:
    ax.bar(lag, acf_vals[lag], color='coral', alpha=0.9)
    ax.annotate(f'k={lag}\n({acf_vals[lag]:.3f})',
                (lag, acf_vals[lag]),
                textcoords="offset points", xytext=(0, 8),
                ha='center', fontsize=8)

ax.set_xlabel('Lag k (days) / 래그 k (일)')
ax.set_ylabel('Autocorrelation / 자기상관')
ax.set_title('F10.7 Autocorrelation Function\nF10.7 자기상관 함수')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Part 7: 롤링 예측 시연 / Rolling Prediction Demonstration

롤링(autoregressive) 예측 모드를 시연합니다. 1-step 모델의 예측값을 입력에 추가하며 반복적으로 60일까지 예측합니다.
이 방식은 단기 예측에서 정확하지만, 오류가 누적되어 장기 예측 성능이 저하됩니다.

We demonstrate the rolling (autoregressive) prediction mode. A 1-step model's prediction is appended to the input and iterated for 60 days.
This approach is accurate short-term, but error accumulation degrades long-horizon performance.

In [ ]:
# Train a 1-step model for rolling prediction
rolling_model = SINet(
    input_len=INPUT_LEN, output_len=1,
    d_model=32, n_blocks=2, top_k=3, dropout=0.3
).to(device)

rolling_dataset = SolarIndexDataset(train_data, INPUT_LEN, output_len=1, mode="rolling")
rolling_loader = DataLoader(rolling_dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer_r = optim.Adam(rolling_model.parameters(), lr=LR)
criterion_r = nn.MSELoss()

print("Training 1-step rolling model...")
for epoch in range(EPOCHS):
    rolling_model.train()
    epoch_loss = 0.0
    n_b = 0
    for x_b, y_b in rolling_loader:
        x_b, y_b = x_b.to(device), y_b.to(device)
        optimizer_r.zero_grad()
        pred = rolling_model(x_b)
        loss = criterion_r(pred, y_b)
        loss.backward()
        optimizer_r.step()
        epoch_loss += loss.item()
        n_b += 1
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:2d}/{EPOCHS}: Loss = {epoch_loss/n_b:.6f}")

# Rolling prediction on a test sample
rolling_model.eval()
test_start = 100  # Pick a starting sample from test set
input_window = torch.tensor(
    test_data[test_start:test_start + INPUT_LEN], dtype=torch.float32
).clone()

rolling_preds = []
with torch.no_grad():
    for step in range(OUTPUT_LEN):
        x_in = input_window.unsqueeze(0).unsqueeze(0).to(device)  # (1, 1, 30)
        pred_step = rolling_model(x_in).item()
        rolling_preds.append(pred_step)
        # Slide window: drop oldest, append prediction
        input_window = torch.cat([input_window[1:], torch.tensor([pred_step])])

# Ground truth for comparison
gt = test_data[test_start + INPUT_LEN : test_start + INPUT_LEN + OUTPUT_LEN]
rolling_preds_sfu = denormalize(np.array(rolling_preds), f107_min, f107_max)
gt_sfu = denormalize(gt, f107_min, f107_max)

# Compare rolling vs fixed prediction for this sample
fixed_pred_sample = preds_sfu[test_start]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(range(1, OUTPUT_LEN + 1), gt_sfu, 'k-', linewidth=1.5, label='Observed / 관측')
ax.plot(range(1, OUTPUT_LEN + 1), fixed_pred_sample, '--', color='steelblue',
        linewidth=1.2, label='Fixed prediction / 고정 예측')
ax.plot(range(1, OUTPUT_LEN + 1), rolling_preds_sfu, '--', color='coral',
        linewidth=1.2, label='Rolling prediction / 롤링 예측')
ax.set_xlabel('Forecast Horizon (days) / 예측 수평선 (일)')
ax.set_ylabel('F10.7 (sfu)')
ax.set_title('Fixed vs Rolling Prediction Comparison\n고정 vs 롤링 예측 비교')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

m_fixed = compute_metrics(gt_sfu, fixed_pred_sample)
m_rolling = compute_metrics(gt_sfu, rolling_preds_sfu)
print(f"Fixed   — RMSE: {m_fixed['rmse']:.2f}, MAE: {m_fixed['mae']:.2f}")
print(f"Rolling — RMSE: {m_rolling['rmse']:.2f}, MAE: {m_rolling['mae']:.2f}")

## 요약 / Summary

### 논문 보고 결과 vs 우리의 구현 / Paper's Reported Results vs Our Implementation

아래 표는 논문의 실제 F10.7 데이터 결과와 우리의 합성 데이터 결과를 비교합니다.
합성 데이터는 실제 데이터보다 단순하므로, 절대 수치보다는 **수평선이 길어질수록 오류가 증가하는 경향**이 핵심입니다.

The table below compares the paper's results on real F10.7 data with our results on synthetic data.
Since synthetic data is simpler than real data, the key takeaway is the **trend of increasing error with longer horizons**, not the absolute numbers.

| 수평선 / Horizon | 논문 RMSE (sfu) | 논문 MAE (sfu) | 논문 MAPE (%) | 비고 / Note |
|---|---|---|---|---|
| d+1 | ~5.2 | ~3.8 | ~3.2 | Highest autocorrelation / 최고 자기상관 |
| d+27 | ~14.5 | ~10.8 | ~8.9 | Solar rotation period / 태양 자전 주기 |
| d+45 | ~18.3 | ~14.1 | ~11.6 | Mid-range forecast / 중기 예측 |
| d+60 | ~20.7 | ~16.0 | ~13.2 | Longest horizon / 최장 수평선 |

*논문의 수치는 근사치이며, 실제 값은 solar cycle phase와 test set에 따라 달라집니다.*
*Paper values are approximate; actual numbers vary with solar cycle phase and test set.*

### 핵심 교훈 / Key Lessons

1. **FFT 기반 주기 탐지** — 시계열의 지배적 주기를 자동으로 추출하여 2D convolution에 활용하는 참신한 접근법
   FFT-based period detection automatically extracts dominant periods for use in 2D convolution — a novel approach.

2. **Dual-Inception Block** — 다양한 커널 크기의 병렬 convolution으로 다중 스케일 패턴을 포착
   Parallel convolutions with diverse kernel sizes capture multi-scale patterns.

3. **고정 vs 롤링** — 고정 예측이 장기 수평선에서 오류 누적을 방지하여 더 안정적
   Fixed prediction avoids error accumulation at long horizons, yielding more stable forecasts.

4. **자기상관과 예측 난이도** — 래그 1에서 자기상관이 가장 높아 단기 예측이 용이하며, 래그가 증가하면 예측 난이도가 상승
   Autocorrelation is highest at lag 1, making short-term prediction easiest; difficulty increases with lag.